# Simple Microscope AI Agent

You have a toy microscope with two knobs — **focus** and **stigmation** — and a hidden optimum.
When both knobs are set to the optimum, sharpness = 1.0. Everywhere else it drops off:

```
sharpness = exp( -(focus - true_focus)² - (stigmation - true_stig)² )
```

Your job: design **tools** and a **system prompt** so that an AI agent can find the optimum within the least amount of measurements.

The microscope and the agent runner are provided. Everything else is up to you.


In [ ]:
!pip install 'strands-agents[openai]' -q

# on windows machines it might be
# %pip install strands-agents[openai] -q

In [ ]:
import logging
import math
import os
from typing import Any

from strands import Agent, tool
from strands.models.openai import OpenAIModel

In [ ]:
# configure log output
logging.getLogger("strands").setLevel(logging.INFO)
logging.getLogger("strands.models").setLevel(logging.DEBUG)
logging.getLogger("strands.tools.executors").setLevel(logging.DEBUG)

In [ ]:
base_url = "https://chat-ai.academiccloud.de/v1"

model_id = "qwen3.5-122b-a10b" # try different models

In [ ]:
#@title API Key Configuration { display-mode: "form" }
# choose one option to set your API key

# 1. Set it directly here and hide the cell content once executed to keep your key private
api_key = "123"

# 2. Read it from an ENV
# api_key = os.environ.get("GWDG_API_KEY")

## The Microscope (given — don't modify)

This is your "hardware." Read through it to understand what's available.


In [ ]:
class Microscope:
    RANGE = (-5.0, 5.0)

    def __init__(self, true_focus: float, true_stig: float, budget: int = None):
        self.true_focus = true_focus
        self.true_stig = true_stig
        self.budget = budget
        self.reset()

    def reset(self):
        self.focus = 0.0
        self.stig = 0.0
        self.history: list[dict[str, float]] = []

    @property
    def used(self) -> int:
        return len(self.history)

    @property
    def remaining(self) -> int:
        return self.budget - self.used

    def clamp(self, v: float) -> float:
        return max(self.RANGE[0], min(self.RANGE[1], float(v)))

    def set_focus(self, value: float) -> float:
        self.focus = self.clamp(value)
        return self.focus

    def set_stig(self, value: float) -> float:
        self.stig = self.clamp(value)
        return self.stig

    def measure(self) -> dict[str, Any]:
        if self.budget and self.remaining <= 0:
            return {"status": "error", "message": f"Budget exhausted ({self.budget} measurements)."}
        sharpness = math.exp(
            -((self.focus - self.true_focus) ** 2 + (self.stig - self.true_stig) ** 2)
        )
        entry = {
            "measurement": self.used + 1,
            "remaining": self.remaining - 1,
            "focus": self.focus,
            "stigmation": self.stig,
            "sharpness": sharpness,
        }
        self.history.append(entry)
        return entry

    def best(self) -> dict[str, float] | None:
        return max(self.history, key=lambda m: m["sharpness"]) if self.history else None


In [ ]:
microscope = Microscope(true_focus=2.3, true_stig=-0.7)

In [ ]:
def print_summary(microscope: Microscope, best):
  print(f"\n{'=' * 50}")
  print(f"measurements used:  {microscope.used} / {microscope.budget}")
  if best:
      print(f"best sharpness:     {best['sharpness']:.10f}")
      print(f"best point:         focus={best['focus']:.4f}  stig={best['stigmation']:.4f}")
  print(f"distance to truth:  focus={microscope.focus - microscope.true_focus:+.4f}  stig={microscope.stig - microscope.true_stig:+.4f}")
  print(f"{'=' * 50}")

## The Runner (given — don't modify)

Call `run(tools, system_prompt)` with your tools and prompt. It handles the agent setup and prints a summary.


In [ ]:
def run_agent(prompt: str, tools: list, system_prompt: str, model_id: str, api_key: str):
    microscope.reset()
    model = OpenAIModel(
        client_args={"api_key": api_key, "base_url": base_url},
        model_id=model_id,
        params={"temperature": 0.0},
    )
    agent = Agent(model=model, system_prompt=system_prompt, tools=tools)

    print(f"model={model_id}  budget={microscope.budget}")
    print(f"(hidden optimum: focus={microscope.true_focus}, stig={microscope.true_stig})\n")

    response = agent(prompt)

    # snap to the best point the agent found
    best = microscope.best()
    if best:
        microscope.focus, microscope.stig = best["focus"], best["stigmation"]

    print_summary(microscope, best)
    return response


---
## Your Turn

### Exercise 1: Define your tools

A tool is a Python function decorated with `@tool`. The agent sees the function name, the docstring, and the argument names/types. It calls the function and gets back whatever you return.

Example pattern:

```python
@tool
def my_tool(arg1: float, arg2: float) -> dict[str, Any]:
    """This docstring is what the agent reads to decide when/how to use this tool."""
    # do something with mic
    return {"result": ...}
```

The microscope gives you: `microscope.set_focus(v)`, `microscope.set_stig(v)`, `microscope.measure()`, `microscope.history`, `microscope.best()`.

**Start simple. You can always add more tools later.**


In [ ]:
# Define your tools here



### Exercise 2: Write your system prompt

This is the instruction the agent reads before it starts. Tell it what the task is, what tools it has, and how to use them.


In [ ]:
SYSTEM_PROMPT = """

"""


### Exercise 3: Write your prompt

This is the instruction that tells the agent which task it should solve.

In [ ]:
PROMPT = """

"""

### Exercise 4: Run it

Pass your tools and prompt to `run()`. Watch the agent work. Then iterate.


In [ ]:
# response = run_agent(PROMPT, [your_tool_1, your_tool_2, ...], SYSTEM_PROMPT, model_id, api_key)


---
### Things to try

After your first run, think about:

- Did the agent waste measurements? Why?
- Did it forget to do something? Could your tool design prevent that?
- Is the agent repeating points it already tried? Could you give it a tool to check?
- Could you encode a smarter search strategy as a tool instead of hoping the agent invents one?
- How much does the docstring matter? Try changing it and re-running.
